# IELTS Listening — Generator QLoRA fine-tune (Kaggle)

Fine-tunes **Qwen2.5-3B-Instruct** with **QLoRA / SFT** to generate the
full doc contract from a spec:

> Blueprint -> Dialogue -> Audio Performance Instructions -> Questions ->
> Official Answers -> Accepted Variants -> Evaluation Metadata

(the *Core Generator Model* + *Training Objective* of
`AI IELTS Listening Exam Engine.md`).

> ⚠️ **Do a clean Restart & Run All** (kernel menu → *Restart & Run All*, or
> a fresh *Save Version*). Do **not** re-run single cells in a kernel that
> already trained — a prior run's 4-bit model + optimizer stay resident on
> the GPU and cause a misleading CUDA OOM at `trainer.train()` even though
> it fits fresh. The load cell now frees leftovers as a guard, but a real
> restart is the reliable path (env-var GPU pinning only applies before CUDA
> initialises).

**Run cell 0 first** to clone the repo (it has the SFT datasets), then
Run All.

### Before you run
1. Notebook settings: **Accelerator = GPU T4** (a *single* T4 — see below)
   and **Internet = On** (needed for the clone + the Unsloth install).
   Two traps: (a) **pick single *T4*, not *T4 x2***. Unsloth's free build
   trains on one GPU, and if two are visible it loads under an accelerate
   device_map dispatch that OOMs GPU 0 — the load cell now forces one GPU
   via `CUDA_VISIBLE_DEVICES=0` as a guard, but pick single T4 anyway. And
   (b) it needs a Turing+ card, so do **not** pick *P100*: it's Pascal
   (compute capability 6.0), Unsloth/Triton have no kernels for it, and it
   dies with `no kernel image is available for execution on the device`. So
   **a single T4 is the target** — the generator's 3B base @ 8192 fits it
   with headroom (7B and 14B both OOM on this long corpus; see section 2).
2. **Get the data** — run **cell 0** to `git clone` this repo (default), or
   add `backend/data/datasets/generator_sft.jsonl` as a **Kaggle Dataset**
   named `ielts-listening-sft`. Cell 3 locates the jsonl either way.

The dataset is chat-format (`{messages:[system,user,assistant]}`); the
assistant turn is the exact JSON the backend already parses, so the
fine-tuned model is a drop-in for the hosted teacher.

## 0. Get the repo + datasets

In [8]:
# Cell 0 — pull this repo so the SFT datasets are on the Kaggle filesystem.
# This is a PRIVATE repo, so give Kaggle a token: Add-ons -> Secrets, add
# GITHUB_TOKEN = a PAT with read access to the repo (fine-grained 'Contents:
# read', or a classic token with the 'repo' scope).
# Alternative: skip cloning and add the 'ielts-listening-sft' Kaggle Dataset;
# the data cell below finds the jsonl either way.
import os
REPO_URL = "https://github.com/asiralabi/IELTS.git"
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{_tok}@")
except Exception:
    print("No GITHUB_TOKEN secret - trying anonymous clone "
          "(only works if the repo is public).")
if not os.path.isdir("ielts"):
    !git clone --depth 1 $REPO_URL ielts
!ls -lh ielts/backend/data/datasets/*.jsonl

No GITHUB_TOKEN secret - trying anonymous clone (only works if the repo is public).
-rw-r--r-- 1 root root 1.7M Jul 21 11:15 ielts/backend/data/datasets/cambridge_listening.jsonl
-rw-r--r-- 1 root root  11M Jul 21 11:15 ielts/backend/data/datasets/evaluator_sft.jsonl
-rw-r--r-- 1 root root 4.9M Jul 21 11:15 ielts/backend/data/datasets/generator_sft.jsonl


In [9]:
%%capture
# Unsloth gives ~2x faster QLoRA. Two Kaggle-specific gotchas: (1) the free
# Unsloth build trains on ONE GPU only, and (2) it needs a Turing-or-newer
# card — use a **T4** (compute capability 7.5). The P100 is Pascal (6.0) and
# has NO compiled Unsloth/Triton kernels -> 'no kernel image is available'.
# Fit depends on the model AND the corpus length: the generator's records
# are long (~5-7k tokens each), so it uses a 3B base; a 7B/14B OOMs a T4 at
# train time on that corpus (see the generator's section 2 for the math).
# If this install ever breaks, follow the current Kaggle snippet at
# https://github.com/unslothai/unsloth (the API used below is stable).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 1. Load Qwen2.5-3B in 4-bit

In [10]:
import os
# Set BEFORE the torch/unsloth CUDA import. Lets the allocator grow
# segments instead of failing on a large contiguous request — cheap VRAM
# hygiene that reduces fragmentation-driven OOMs.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Pin to ONE GPU. Unsloth's free build trains on a single GPU anyway, but
# if TWO are visible (e.g. you picked 'T4 x2') it loads under an accelerate
# device_map dispatch whose per-forward hooks pile extra tensors onto GPU 0
# and OOM it. One visible GPU = clean single-device load. Set before import.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc, torch
# Reclaim VRAM left by a PREVIOUS run in this kernel. If you re-run cells
# without restarting, the old 4-bit model + optimizer stay resident on the
# GPU and cause a misleading OOM at train time even though the model fits
# fresh. A clean Restart & Run All avoids this; this guard makes a reused
# kernel non-fatal by freeing those globals first.
for _n in ("model", "tokenizer", "trainer", "trainer_stats"):
    globals().pop(_n, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from unsloth import FastLanguageModel

MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto: bf16 on Ampere+, fp16 on T4
    load_in_4bit=True,   # QLoRA
    # Pin the whole model to GPU 0: no cross-GPU sharding, and the model
    # device matches the trainer device. A bnb 4-bit model loaded on a
    # different device than the trainer makes accelerate raise ValueError
    # ('can't train a model loaded in 4-bit precision on a different
    # device...') at trainer.train(); this is the fix it recommends.
    device_map={"": 0},
)

==((====))==  Unsloth 2026.7.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## 2. Attach LoRA adapters

The generator records are long and **uniformly** so — measured on the real
corpus every record is 4.6k-6.9k tokens (a ~2.8k-token system prompt + the
full exam JSON), median ~5.9k. So `MAX_SEQ_LEN` **must stay >= 7168** (8192
is the safe value used here) or the assistant JSON gets truncated on the
longest records and the model learns to emit incomplete exams. Do **not**
lower it to save VRAM — and because the *shortest* record is already 4.6k,
dropping the long samples doesn't help either (a 6144 cap keeps 185/241 but
barely dents VRAM; a 4096 cap keeps 0).

### Why 3B, not 7B or 14B
The doc's Core Generator is nominally 14B, but on a Kaggle T4 (15 GB, the
only usable free card — P100 is Pascal and can't run Unsloth) **neither 14B
nor 7B fits this corpus**:
- **14B** loads fine, then OOMs at the first attention forward in
  `trainer.train()` (~40 MB short); `expandable_segments` can't recover it.
- **7B** loads and even starts, then OOMs in the *backward* pass: on a
  genuinely clean single-T4 run PyTorch holds ~14.3 GiB before a 2.79 GiB
  attention-gradient alloc for a ~6.9k-token sample — ~2.7 GiB short, with
  no fragmentation slack to reclaim. The O(seq^2) attention is the killer,
  and since every sample is 5-7k tokens you can't shrink it without gutting
  the corpus.

So the generator uses **Qwen2.5-3B**, which keeps all 241 records at the
full 8192 context (no truncation) and fits the T4 with headroom (~4 GiB
freed vs 7B: a smaller 4-bit model plus a smaller attention matrix). If a
clean run still OOMs, drop to `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit`. To
train 7B/14B instead, use a >=24 GB Turing+ GPU off-Kaggle (A10/L4/3090)
and set `MODEL` back accordingly.

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",   # long scripts -> save VRAM
    random_state=3407,
)

Unsloth 2026.7.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 3. Load the SFT dataset

In [12]:
import glob, os
from datasets import load_dataset

# Resolves the SFT jsonl produced by backend/tools/build_dataset.py.
# Works whether you git-clone this repo into the notebook OR add it as a
# Kaggle Dataset named 'ielts-listening-sft'.
FILENAME = "generator_sft.jsonl"
CANDIDATES = [
    f"/kaggle/input/ielts-listening-sft/{FILENAME}",
    *glob.glob(f"/kaggle/**/backend/data/datasets/{FILENAME}", recursive=True),
    *glob.glob(f"**/backend/data/datasets/{FILENAME}", recursive=True),
]
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"{FILENAME} not found. Git-clone this repo (see cell 0) or add the "
        "Kaggle Dataset 'ielts-listening-sft'.")
print("Using dataset:", DATA_PATH)

raw = load_dataset("json", data_files=DATA_PATH, split="train")

def to_text(row):
    # Each row is {"messages": [system, user, assistant]}; render with
    # the model's own chat template so training matches inference.
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)}

dataset = raw.map(to_text, remove_columns=raw.column_names)
print(dataset)
print(dataset[0]["text"][:600])

Using dataset: /kaggle/working/ielts/backend/data/datasets/generator_sft.jsonl


Map:   0%|          | 0/241 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 241
})
<|im_start|>system
You are an IELTS Listening test writer. Generate a complete, exam-authentic practice set built around a listening script.

Work in this order (the official design pipeline): first a Blueprint, then the Dialogue, then the Audio Performance Instructions, then the Questions, the Official Answers, the Accepted Variants, and finally the Evaluation Metadata.

Blueprint — REQUIRED `blueprint` object:
- Before writing the script, design the exam on paper. Add a top-level `blueprint` object:
  {
    "section": "<Part 1|Part 2|Part 3|Part 4>",
    "topic": "<the scenario in a few word


## 4. Train (response-only loss)

In [13]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=8192,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="qwen2.5-3b-ielts-generator-lora-checkpoints",
        report_to="none",
    ),
)

# Mask the prompt: only the assistant JSON contributes to the loss, so
# the model learns to PRODUCE exams, not to echo the spec/instructions.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/241 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/241 [00:00<?, ? examples/s]

In [14]:
trainer_stats = trainer.train()
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 241 | Num Epochs = 3 | Total steps = 93
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


OutOfMemoryError: CUDA out of memory. Tried to allocate 410.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 144.81 MiB is free. Including non-PyTorch memory, this process has 14.42 GiB memory in use. Of the allocated memory 14.21 GiB is allocated by PyTorch, and 57.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 5. Save adapter + GGUF

In [ ]:
# 1) LoRA adapter only (tiny, ~100-300 MB) — load on top of the base.
model.save_pretrained("qwen2.5-3b-ielts-generator-lora")
tokenizer.save_pretrained("qwen2.5-3b-ielts-generator-lora")

# 2) GGUF (q4_k_m) for llama.cpp / Ollama serving on CPU or small GPU.
#    Merges + quantises; needs several GB of /kaggle/working disk.
model.save_pretrained_gguf("qwen2.5-3b-ielts-generator-gguf", tokenizer, quantization_method="q4_k_m")

# 3) (optional) merged 16-bit HF weights for vLLM (~6 GB for 3B, ~14 GB for
#    7B) — uncomment
#    only if you have the disk / will push to the HF Hub.
# model.save_pretrained_merged("qwen2.5-3b-ielts-generator-lora-merged16", tokenizer, save_method="merged_16bit")

## 6. Serve it, and point the backend at it

**Option A — Ollama (local, CPU-friendly).** Download the GGUF, then:
```
# Modelfile
FROM ./qwen2.5-3b-ielts-generator-gguf/unsloth.Q4_K_M.gguf
PARAMETER temperature 0.4
PARAMETER num_ctx 8192
```
```
ollama create ielts-generator -f Modelfile
```
Then in `backend/.env`:
```
LLM_PROVIDER=ollama
OLLAMA_MODEL=ielts-generator
```

**Option B — vLLM (GPU, OpenAI-compatible).** Serve the merged weights
(or base + adapter) and set:
```
LLM_PROVIDER=openai
OPENAI_BASE_URL=http://<host>:8000/v1
OPENAI_MODEL=ielts-generator
OPENAI_API_KEY=dummy
```
No app code changes are needed — `app/llm/client.py` already speaks both.